# Manuscript Fig. 5 — policy analysis

Current formal plotting cells are collected here in manuscript panel order.

## Cell organization

Panels follow the manuscript order; existing composite groups remain combined.
Run the setup cell before rendering panels. The shared export uses 8 pt text by default.
Exports use `Fig{number}{panel(s)}_{description}` in `figure/`: PNG (600 dpi), PDF, and SVG with text converted to paths.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np


start_dir = Path.cwd().resolve()
workbench_dir = next(
    (candidate
     for root in (start_dir, *start_dir.parents)
     for candidate in (root, root / "figure_workbench", root / "DiFCoN" / "figure_workbench")
     if (candidate / "plotting" / "__init__.py").is_file()),
    None,
)
if workbench_dir is None:
    raise FileNotFoundError("Cannot locate figure_workbench/plotting from the current directory")
if str(workbench_dir) not in sys.path:
    sys.path.insert(0, str(workbench_dir))

from plotting import subfigure_plotting_utils as sp
from plotting._paths import FIGURE_DIR
from plotting._source_tables import read_source_table
from plotting.manuscript_export import DEFAULT_FONT_SIZES as FONT_SIZES, save_main_panel
from plotting import policy_robustness_plotting as prp

FONT_SIZE = FONT_SIZES["base"]
print(f"Output root: {FIGURE_DIR}")

## Fig. 5a — dose-efficacy trade-off

In [ ]:
weight_font_sizes = {**FONT_SIZES, 'legend': 7}
FIGSIZE = (100 / 25.4, 65 / 25.4)

# ============================================================
# Load the saved source tables from the analysis.
# ============================================================

weight_summary = read_source_table('dose_weight', 'cohort_summary').sort_values("total_dose_weight")

fixed_test = read_source_table('dose_weight', 'fixed_control_comparison')

# Colors from the reference figure
WEIGHT_COLORS = {
    0.1: "#3F88A8",
    0.3: "#86B6C8",
    0.5: "#6FA37B",
    0.75: "#9EBB7D",
    1.0: "#D2A13B",
    1.5: "#D97845",
    2.0: "#B64B5C",
}
COMBINE_COLOR = "#608F70"

# ============================================================
# Read policy test-set means and SDs across seeds for λd > 0.
# ============================================================

data = weight_summary[weight_summary["total_dose_weight"] > 0].copy()

weights = data["total_dose_weight"].to_numpy(float)
dose = data["test_total_dose_mean"].to_numpy(float)
tumor = data["test_final_C_rel_mean"].to_numpy(float)
dose_sd = data["test_total_dose_std"].to_numpy(float)
tumor_sd = data["test_final_C_rel_std"].to_numpy(float)

# Compute test cohort means for the fixed Combination regimen.
combine = (
    fixed_test.loc[fixed_test["group"].eq("Combine"), ["total_dose", "final_C_rel"]]
    .mean()
)

# ============================================================
# Main figure panel
# ============================================================

with sp.figure_style(FONT_SIZE):
    fig, ax = plt.subplots(figsize=FIGSIZE)

    # Connect the dose and efficacy results across λd values.
    ax.plot(dose, tumor, color="#AEB4BC", lw=0.75, zorder=1)

    # For each λd, show the mean and SD across the three seeds.
    for w, x, y, xerr, yerr in zip(weights, dose, tumor, dose_sd, tumor_sd):
        ax.errorbar(
            x, y, xerr=xerr, yerr=yerr, fmt="none",
            ecolor="#8C929A", elinewidth=0.7, capsize=1.8, zorder=2
        )
        ax.scatter(
            x, y, s=42, color=WEIGHT_COLORS[w],
            edgecolor="white", linewidth=0.55,
            label=rf"$w_d={w:g}$", zorder=3
        )

    # Fixed Combination baseline
    ax.scatter(
        combine["total_dose"], combine["final_C_rel"],
        marker="D", s=46, color=COMBINE_COLOR,
        edgecolor="white", linewidth=0.6,
        label="Combination", zorder=4
    )

    # Set the main panel axis labels.
    ax.set_xlabel(r"Mean total dose, $\bar{D}_{\mathrm{total}}$", fontsize=weight_font_sizes["axis_label"])
    ax.set_ylabel(r"Mean final tumor, $C(T)/C(0)$", fontsize=weight_font_sizes["axis_label"])

    # Use axis limits close to those of the reference figure.
    ax.set_xlim(float(np.min(dose)) - 0.08,max(float(np.max(dose)), float(combine["total_dose"])) + 0.20)

    # Hide the grid for the main figure.
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # λd + Combination
    handles, labels = ax.get_legend_handles_labels()
    desired_labels = [
        r"$\widetilde{w}_d=0.1$", r"$\widetilde{w}_d=0.3$",
        r"$\widetilde{w}_d=0.5$", r"$\widetilde{w}_d=0.75$",
        r"$\widetilde{w}_d=1$", r"$\widetilde{w}_d=1.5$",
        r"$\widetilde{w}_d=2$", "Combination"
    ]
    by_label = {
        label.replace("w_d", r"\widetilde{w}_d"): handle
        for label, handle in zip(labels, handles)
    }
    handles = [by_label[x] for x in desired_labels]
    labels = desired_labels


    # Reorder the entries for Matplotlib's column-first legend layout.
    order = [0, 4, 1, 5, 2, 6, 3, 7]
    handles = [handles[i] for i in order]
    labels = [labels[i] for i in order]

    ax.legend(
        handles, labels, loc="upper right", ncol=4,
        frameon=False, fontsize=weight_font_sizes["legend"],
        columnspacing=0.25, handletextpad=0.3,
        labelspacing=0.75, borderaxespad=0.2
    , title_fontsize=weight_font_sizes["legend_title"])

    fig.tight_layout()

paths = save_main_panel(fig, "Fig5a_dose_efficacy_tradeoff", font_sizes=weight_font_sizes)

print(paths)
fig

## Fig. 5b — loss-component ablation

In [ ]:
FIGSIZE = (100 / 25.4, 65 / 25.4)

fig = sp.seed2031_loss_ablation_bar_panel(figsize=FIGSIZE,font_size=FONT_SIZE,)

paths = save_main_panel(fig, "Fig5b_loss_component_ablation")

print(paths)
fig

## Fig. 5c — open-loop versus closed-loop outcomes

In [ ]:
FIGSIZE = (100 / 25.4, 58 / 25.4)
fig = sp.plot_openloop_control_comparison(figsize=FIGSIZE, font_size=FONT_SIZE)

paths = save_main_panel(fig, "Fig5c_openloop_vs_closedloop_outcomes", bbox_inches=None)
print(paths)
fig

## Fig. 5d — feedback dose-control trade-off

In [ ]:
FIGSIZE = (100 / 25.4, 60 / 25.4)

fig = sp.openloop_feedback_tradeoff_panel(figsize=FIGSIZE, font_size=FONT_SIZE)
paths = save_main_panel(fig, "Fig5d_feedback_dose_control_tradeoff")

print(paths)
fig

## Fig. 5e — full versus practical observability

In [ ]:
FIGSIZE = (100 / 25.4, 58 / 25.4)
fig = sp.plot_observation_full_vs_practical(figsize=FIGSIZE, font_size=FONT_SIZE)

paths = save_main_panel(fig, "Fig5e_full_vs_practical_observability", bbox_inches=None)
print(paths)
fig

## Fig. 5f-g — temporal attribution and history perturbation

Panels f and g are generated together by the single existing plotting cell below and are intentionally not split.

In [ ]:
lag = read_source_table('attribution', 'lag_attribution')
patient = read_source_table('attribution', 'intervention_patient_means')
stats = read_source_table('attribution', 'intervention_statistics')

fig = prp.plot_temporal_attribution_summary(
    lag, patient, stats,
    figsize=(100 / 25.4, 58 / 25.4),
    font_size=FONT_SIZE,
)
paths = save_main_panel(fig, "Fig5f-g_temporal_attribution_history_perturbation")
print(paths)
fig